# Fed-batch bioprocess simulator: worked examples

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ftwigg/bioprocess_sim/blob/main/notebooks/fedbatch_simulator_examples.ipynb)

Three complete fed-batch runs with plots, plus two comparisons:

1. **Exponential fed-batch on glucose**: batch, exponential feed sized from the biomass present, feed stop.
2. **DO-stat fed-batch**: oxygen supply fixed at maximum, the feed pump alone holds dissolved oxygen.
3. **Glycerol instead of glucose**: same model, different carbon source, stoichiometry recalculated.
4. **Glucose against glycerol**, side by side.
5. **Overfeeding**: what the oxygen-transfer ceiling does to an open-loop exponential feed.

It works in Google Colab (the first cell clones the repository) and in a local Jupyter session started from a clone. The whole notebook runs in about a minute.

## Setup

In [ ]:
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/ftwigg/bioprocess_sim.git"
BRANCH = "main"   # the branch this notebook clones when run in Colab


def locate_repo():
    """Use the checkout this notebook lives in, or clone one next to it (Colab)."""
    here = pathlib.Path.cwd()
    for folder in [here, *here.parents]:
        if (folder / "examples" / "_bootstrap.py").exists():
            return folder
    target = here / "bioprocess_sim"
    if target.exists() and not (target / "examples" / "_bootstrap.py").exists():
        raise RuntimeError(
            f"{target} exists but is not the {BRANCH} branch. "
            "Delete that folder and run this cell again.")
    if not target.exists():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", BRANCH,
                        REPO_URL, str(target)], check=True)
    return target


REPO = locate_repo()
sys.path.insert(0, str(REPO / "examples"))
import _bootstrap  # noqa: F401  registers the repository as the package `fedbatch_simulator`

import fedbatch_simulator
print(f"Simulator {fedbatch_simulator.__version__} loaded from {REPO}")

In [ ]:
import numpy as np
import scipy
import matplotlib
import matplotlib.pyplot as plt

from fedbatch_simulator import *
from fedbatch_simulator.kinetics import CellParameters
from fedbatch_simulator.mass_transfer import DynamicKLa
from fedbatch_simulator.thermodynamics import TemperatureController
from fedbatch_simulator.feed_strategies import ConstantFeed, ExponentialFeed, DOStatFeed, PiecewiseFeed
from fedbatch_simulator.control import PICascade, SimplifiedCascade, FixedAerationAgitation
from fedbatch_simulator.reactor import FedBatchReactor
from fedbatch_simulator.simulator import FedBatchSimulator
from fedbatch_simulator.plotting import plot_fedbatch_results, plot_comparison, do_saturation

DO_SAT = do_saturation(37.0)   # mmol/L in equilibrium with air at 37°C
print(f"Python {sys.version.split()[0]}, NumPy {np.__version__}, SciPy {scipy.__version__}, "
      f"Matplotlib {matplotlib.__version__}")
print(f"DO at air saturation: {DO_SAT:.4f} mmol/L")

---

# Example 1: exponential fed-batch on glucose

An *E. coli*-like culture in a 5 L lab reactor.

- **0 to 5.5 h, batch.** 10 g/L of glucose runs out at about 5.3 h.
- **5.5 to 22 h, feed.** Exponential at 0.12 1/h until it reaches a cap of 4.5 g glucose/h, then constant.
- **22 to 26 h, feed off.** Starvation and slow endogenous decay.

### Cells

Oxygen yield, oxygen maintenance and RQ are not inputs. They follow from the electron and carbon balances for the chosen substrate, biomass composition, yield and maintenance.

In [ ]:
carbon_params = SubstrateParameters(
    name='carbon',
    Ks=0.1,     # g/L
    Y_xs=0.5,   # g biomass / g glucose (true yield)
    ms=0.03     # g glucose / g biomass / h (maintenance)
)
nitrogen_params = SubstrateParameters(name='nitrogen', Ks=0.05, Y_xs=10.0, ms=0.001)

cell_params = CellParameters(
    mu_max=0.5,                       # 1/h
    carbon_source=GLUCOSE,
    biomass_composition=STANDARD_BIOMASS,
    substrates={'carbon': carbon_params, 'nitrogen': nitrogen_params},
    Y_xs=0.5,
    Ks_O2=0.003                       # mmol/L
)

print(f"mu_max:        {cell_params.mu_max} 1/h")
print(f"Y_x/S:         {cell_params.Y_xs} g/g")
print(f"Y_x/O2:        {cell_params.Y_x_O2:.3f} g/g      (from the electron balance)")
print(f"m_O2:          {cell_params.m_O2:.4f} g/g/h   (from the substrate maintenance)")
print(f"RQ of growth:  {cell_params.RQ:.3f}          (from the carbon balance)")

### Reactor and feed

The feed is sized from two facts:

- **Where to start:** the batch leaves about `X0 + 0.48 × S0` of biomass. `ExponentialFeed.initial_rate` gives the feed rate that biomass needs to grow at the target rate.
- **Where to stop:** the cap comes from oxygen, not from the pump. This reactor transfers roughly 35 mmol O2/L/h at 27% DO once biomass is near 20 g/L, which covers about 4.5 g glucose/h. Example 5 shows what happens without the cap.

In [ ]:
reactor_config = LAB_STR_5L

initial_state = ReactorState(
    X=0.5,            # g/L
    S_carbon=10.0,    # g/L batch glucose
    S_nitrogen=2.0,   # g/L
    DO=0.2,           # mmol/L (about 90% of air saturation)
    DCO2=0.5,         # mmol/L
    V=2.0,            # L
    T=37.0,           # °C
    N=300,            # rpm
    pH=7.0
)

feed_composition = FeedComposition(S_carbon=500.0, S_nitrogen=50.0, temperature=25.0, pH=5.0)

t_feed_start, t_feed_stop = 5.5, 22.0   # h
mu_set = 0.12                           # 1/h
glucose_cap = 4.5                       # g/h

X_batch_end = initial_state.X + 0.48 * initial_state.S_carbon
F0 = ExponentialFeed.initial_rate(mu_set=mu_set, X=X_batch_end, V=initial_state.V,
                                  S_feed=feed_composition.S_carbon,
                                  Y_xs=carbon_params.Y_xs, ms=carbon_params.ms)

batch_feed = ConstantFeed(feed_composition, F_constant=0.0)
exp_feed = ExponentialFeed(feed_composition, F0=F0, mu_set=mu_set,
                           F_max=glucose_cap / feed_composition.S_carbon)
exp_feed.t_start = t_feed_start

feed_strategy = PiecewiseFeed([
    (0.0, batch_feed),
    (t_feed_start, exp_feed),
    (t_feed_stop, batch_feed)
])

print(f"Reactor: {reactor_config.name}, starting volume {initial_state.V} L")
print(f"Feed starts at {F0 * 1000:.2f} mL/h and is capped at "
      f"{glucose_cap / feed_composition.S_carbon * 1000:.2f} mL/h ({glucose_cap} g glucose/h)")

### Controllers and run

`PICascade` is a PI controller with split-range actuation: agitation first, then gas flow. It has finite gain and integral time, so dissolved oxygen overshoots when demand changes quickly.

In [ ]:
def make_cascade(controller=PICascade, DO_setpoint=0.06):
    """DO cascade for the reactor and starting volume defined above."""
    return controller(
        DO_setpoint=DO_setpoint,      # mmol/L, about 27% of air saturation
        N_min=reactor_config.N_min,
        N_max=reactor_config.N_max,
        Q_gas_min=reactor_config.Q_gas_min * initial_state.V * 60,   # L/h
        Q_gas_max=reactor_config.Q_gas_max * initial_state.V * 60,
        kLa_correlation=DynamicKLa(k_O2=reactor_config.k_O2, a=reactor_config.a,
                                   b=reactor_config.b, k_X=reactor_config.k_X))


def make_temperature_controller():
    return TemperatureController(T_setpoint=37.0, K_p=10.0, K_i=0.5)


reactor = FedBatchReactor(
    cell_params=cell_params,
    reactor_config=reactor_config,
    initial_state=initial_state,
    feed_strategy=feed_strategy,
    do_control=make_cascade(),
    temp_controller=make_temperature_controller()
)

results_exp = FedBatchSimulator(reactor).simulate(t_end=26.0, max_step=0.1)

### Results

What to look for:

- **Dissolved oxygen** spikes when the batch glucose runs out and again when the feed stops.
- **Carbon substrate** stays at a few tens of mg/L while feeding (carbon-limited).
- **Agitation** saturates first, then **gas flow** takes over.
- **RQ** is just above 1 while growing and about 0.95 once starved.

In [ ]:
fig = plot_fedbatch_results(results_exp, "Example 1: exponential fed-batch on glucose",
                            DO_setpoint=0.06,
                            marks={'feed on': t_feed_start, 'feed off': t_feed_stop})
plt.show()

In [ ]:
t = results_exp.time
DO_pct = results_exp.DO / DO_SAT * 100
feeding = (t > t_feed_start + 0.5) & (t < t_feed_stop)
i_peak = np.argmax(results_exp.X * results_exp.V)

print(f"Peak biomass:                    {results_exp.X[i_peak] * results_exp.V[i_peak]:.1f} g "
      f"({results_exp.X[i_peak]:.1f} g/L) at {t[i_peak]:.1f} h")
print(f"Volumetric productivity to peak: {results_exp.X[i_peak] / t[i_peak]:.2f} g/L/h")
print(f"DO spike at glucose exhaustion:  {DO_pct[(t > 4.5) & (t < 5.8)].max():.0f}% of saturation")
print(f"DO spike at feed stop:           {DO_pct[t > t_feed_stop].max():.0f}% of saturation")
print(f"DO while feeding:                {DO_pct[feeding].min():.0f}-{DO_pct[feeding].max():.0f}% of saturation")
print(f"Glucose while feeding:           at most {results_exp.S_carbon[feeding].max() * 1000:.0f} mg/L")
print(f"Peak oxygen uptake rate:         {results_exp.OUR.max():.0f} mmol/L/h")
print()
print(format_balances(check_balances(results_exp, cell_params, feed_strategy)))

---

# Example 2: DO-stat fed-batch

Agitation and gas flow are held at maximum for the whole run, and after the batch the feed pump alone controls dissolved oxygen:

- DO above the setpoint means the culture can take more substrate, so feed more.
- DO below the setpoint means it is being overfed, so feed less.

If a DO cascade and a DO-stat feed both act on the same measurement they fight each other. Fixing the oxygen supply at its maximum and letting the feed set the oxygen demand removes the conflict and runs the reactor at its full transfer capacity.

The 10 g/L of batch glucose is chosen so that the batch ends just as oxygen demand reaches what the reactor can transfer.

In [ ]:
N_fixed = reactor_config.N_max                                   # rpm
Q_gas_fixed = reactor_config.Q_gas_max * initial_state.V * 60    # L/h

oxygen_supply = FixedAerationAgitation(
    N=N_fixed, Q_gas=Q_gas_fixed,
    kLa_correlation=DynamicKLa(k_O2=reactor_config.k_O2, a=reactor_config.a,
                               b=reactor_config.b, k_X=reactor_config.k_X))

initial_state_dostat = ReactorState(X=0.5, S_carbon=10.0, S_nitrogen=2.0, DO=0.2, DCO2=0.5,
                                    V=2.0, T=37.0, N=N_fixed, pH=7.0)

t_dostat_start = 5.6   # h, shortly after the batch glucose runs out

do_stat_feed = DOStatFeed(
    feed_composition=feed_composition,
    DO_setpoint=0.06,   # mmol/L
    K_p=0.1,            # L/h per mmol/L; positive: high DO means more feed
    K_i=1.0,            # L/h per mmol/L per h
    F_min=0.0,
    F_max=0.03,         # L/h
    F_base=0.0
)

feed_strategy_dostat = PiecewiseFeed([
    (0.0, batch_feed),
    (t_dostat_start, do_stat_feed)
])

reactor_dostat = FedBatchReactor(
    cell_params=cell_params,
    reactor_config=reactor_config,
    initial_state=initial_state_dostat,
    feed_strategy=feed_strategy_dostat,
    do_control=oxygen_supply,
    temp_controller=make_temperature_controller()
)

results_dostat = FedBatchSimulator(reactor_dostat).simulate(t_end=24.0, max_step=0.1)

What to look for:

- **Batch:** with nothing controlling it, dissolved oxygen falls from about 90% as biomass grows, reaches the setpoint region right at glucose exhaustion, then jumps to about 95%.
- **DO-stat:** the feed pulls dissolved oxygen back to the setpoint and holds it there.
- **Feed rate** declines slowly, because kLa falls as biomass and volume rise.

In [ ]:
fig = plot_fedbatch_results(results_dostat, "Example 2: DO-stat fed-batch, oxygen supply fixed at maximum",
                            DO_setpoint=0.06, marks={'DO-stat on': t_dostat_start})
plt.show()

In [ ]:
t = results_dostat.time
DO_pct = results_dostat.DO / DO_SAT * 100
batch = t < t_dostat_start
settled = t >= t_dostat_start + 1.0

print("Batch phase")
print(f"  Lowest DO, at glucose exhaustion: {DO_pct[batch].min():.0f}% of saturation")
print(f"  DO spike after exhaustion:        {DO_pct[batch].max():.0f}% of saturation")
print(f"DO-stat phase (after {t_dostat_start + 1.0:.1f} h)")
print(f"  DO range:        {DO_pct[settled].min():.1f}-{DO_pct[settled].max():.1f}% of saturation "
      f"(setpoint {0.06 / DO_SAT * 100:.1f}%)")
print(f"  Feed rate range: {results_dostat.F_feed[settled].min() * 1000:.1f}-"
      f"{results_dostat.F_feed[settled].max() * 1000:.1f} mL/h")
print(f"  Glucose:         at most {results_dostat.S_carbon[settled].max() * 1000:.0f} mg/L")
print(f"Final biomass: {results_dostat.X[-1] * results_dostat.V[-1]:.1f} g ({results_dostat.X[-1]:.1f} g/L), "
      f"productivity {results_dostat.X[-1] / t[-1]:.2f} g/L/h")
print()
print(format_balances(check_balances(results_dostat, cell_params, feed_strategy_dostat)))

---

# Example 3: glycerol instead of glucose

Only the `CarbonSource` object and the kinetic constants change. The stoichiometry is recalculated: glycerol is more reduced than glucose, so each gram needs more oxygen and gives a lower RQ.

In [ ]:
print(f"{'':22s} {'glucose':>10} {'glycerol':>10}")
print(f"{'Formula':22s} {GLUCOSE.formula:>10} {GLYCEROL.formula:>10}")
print(f"{'Degree of reduction':22s} {GLUCOSE.degree_of_reduction:10.2f} {GLYCEROL.degree_of_reduction:10.2f}")
for label, fn in [('Y_x/O2 (g/g)', calculate_oxygen_yield), ('RQ of growth', calculate_RQ)]:
    print(f"{label:22s} {fn(GLUCOSE, STANDARD_BIOMASS, 0.5):10.3f} {fn(GLYCEROL, STANDARD_BIOMASS, 0.5):10.3f}")

The same oxygen supply covers less glycerol than glucose, so the feed cap drops from 4.5 to 3 g/h. Growth on glycerol is slower here, so the batch ends later (about 6.8 h) and the feed starts at 7 h.

In [ ]:
glycerol_carbon = SubstrateParameters(name='carbon', Ks=0.15, Y_xs=0.5, ms=0.03)

cell_params_glycerol = CellParameters(
    mu_max=0.4,
    carbon_source=GLYCEROL,
    biomass_composition=STANDARD_BIOMASS,
    substrates={'carbon': glycerol_carbon, 'nitrogen': nitrogen_params},
    Y_xs=0.5,
    Ks_O2=0.003
)

feed_composition_glycerol = FeedComposition(S_carbon=600.0, S_nitrogen=50.0, temperature=25.0, pH=5.0)

t_feed_start_gly, mu_set_gly, glycerol_cap = 7.0, 0.10, 3.0

F0_gly = ExponentialFeed.initial_rate(mu_set=mu_set_gly, X=X_batch_end, V=initial_state.V,
                                      S_feed=feed_composition_glycerol.S_carbon,
                                      Y_xs=glycerol_carbon.Y_xs, ms=glycerol_carbon.ms)
exp_feed_gly = ExponentialFeed(feed_composition_glycerol, F0=F0_gly, mu_set=mu_set_gly,
                               F_max=glycerol_cap / feed_composition_glycerol.S_carbon)
exp_feed_gly.t_start = t_feed_start_gly

no_feed_gly = ConstantFeed(feed_composition_glycerol, F_constant=0.0)
feed_strategy_glycerol = PiecewiseFeed([
    (0.0, no_feed_gly),
    (t_feed_start_gly, exp_feed_gly),
    (t_feed_stop, no_feed_gly)
])

reactor_glycerol = FedBatchReactor(
    cell_params=cell_params_glycerol,
    reactor_config=reactor_config,
    initial_state=initial_state,
    feed_strategy=feed_strategy_glycerol,
    do_control=make_cascade(),
    temp_controller=make_temperature_controller()
)

results_glycerol = FedBatchSimulator(reactor_glycerol).simulate(t_end=26.0, max_step=0.1)

In [ ]:
fig = plot_fedbatch_results(results_glycerol, "Example 3: exponential fed-batch on glycerol",
                            DO_setpoint=0.06,
                            marks={'feed on': t_feed_start_gly, 'feed off': t_feed_stop})
plt.show()

In [ ]:
growing = results_glycerol.mu > 0.05
starving = results_glycerol.time > t_feed_stop + 0.5
rq = results_glycerol.CER / results_glycerol.OUR

print(f"RQ while growing on glycerol: {rq[growing].mean():.3f} "
      f"(growth alone: {cell_params_glycerol.RQ:.3f}; maintenance pulls it up slightly)")
print(f"RQ while starving:            {rq[starving].mean():.3f} "
      f"(the cells are oxidising their own biomass, whatever the substrate was)")
print()
print(format_balances(check_balances(results_glycerol, cell_params_glycerol, feed_strategy_glycerol)))

---

# Example 4: glucose against glycerol

The two exponential runs on shared axes. Both reach a similar oxygen uptake rate, because that is set by the reactor. Glycerol gets there with less substrate and produces less CO2 per O2.

In [ ]:
fig = plot_comparison({'Glucose': results_exp, 'Glycerol': results_glycerol},
                      "Glucose against glycerol in the same reactor")
plt.show()

---

# Example 5: overfeeding

The same model with an open-loop exponential feed that ignores the oxygen ceiling: 10 mL/h at 10 h, growing at 0.2 1/h, with a pump limit far above anything the culture can use.

Oxygen demand outruns transfer at about 13 h. From then on agitation and gas flow are at their limits, dissolved oxygen is near zero, oxygen uptake is pinned to the transfer rate, and the glucose that cannot be burned accumulates.

A real culture would respond with overflow metabolism (acetate) and inhibition, which this model does not include. What it does show is the ceiling itself, with nothing going negative and the balances still closing.

In [ ]:
open_loop = ExponentialFeed(feed_composition, F0=0.01, mu_set=0.2, F_max=0.5)
open_loop.t_start = 10.0
feed_strategy_over = PiecewiseFeed([(0.0, batch_feed), (10.0, open_loop)])

reactor_over = FedBatchReactor(
    cell_params=cell_params,
    reactor_config=reactor_config,
    initial_state=initial_state,
    feed_strategy=feed_strategy_over,
    do_control=make_cascade(),
    temp_controller=make_temperature_controller()
)

results_over = FedBatchSimulator(reactor_over).simulate(t_end=24.0, max_step=0.1)

fig = plot_fedbatch_results(results_over, "Example 5: open-loop exponential feed beyond the oxygen ceiling",
                            DO_setpoint=0.06, marks={'feed on': 10.0})
plt.show()

In [ ]:
late = results_over.time > 18.0
report_over = check_balances(results_over, cell_params, feed_strategy_over)

print(f"Glucose at the end:      {results_over.S_carbon[-1]:.0f} g/L")
print(f"DO after 18 h:           at most {results_over.DO[late].max() / DO_SAT * 100:.1f}% of saturation")
print(f"OUR / OTR after 18 h:    {np.mean(results_over.OUR[late] / results_over.OTR[late]):.3f} "
      f"(uptake pinned to transfer)")
print(f"Biomass at the end:      {results_over.X[-1]:.1f} g/L")
print()
print(format_balances(report_over))

---

# Where to go next

- Change a number in any cell above and re-run from there. `mu_set`, the feed cap, the DO setpoint and the controller tuning (`K_c`, `tau_i` on `PICascade`) are the most instructive.
- Swap `make_cascade()` for `make_cascade(SimplifiedCascade)` to see the idealised controller, which never leaves the setpoint.
- `LAB_STR_5L` is one of several reactor presets in `base_models.py`. Note that the kLa correlation uses rpm directly and does not yet scale to the larger presets.
- The same three runs exist as plain scripts in `examples/`, and `tests/test_physical_sanity.py` checks the behaviour shown here.